# Public Rainfall Distribution Analysis for Rainwater Harvesting

This notebook is a simplified, public-use version of the rainfall distribution workflow. It loads CHIRPS rainfall data, extracts pixels inside any user-provided area of interest, fits a small set of distributions, and exports rainfall quantiles for downstream RWH analysis.

Compared with the private notebook, this version removes hardcoded local paths, custom distribution code, progress tracking, and country-specific assumptions.

## Inputs

Set the paths below to your own data locations before running the notebook.

Expected inputs:
- An area-of-interest shapefile or GeoPackage in EPSG:4326
- A folder of CHIRPS daily NetCDF files

Outputs are saved as CSV files in the output folder.

In [ ]:
# 1. Imports and configuration
from pathlib import Path
import os
import glob
import json
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point
import scipy.stats as stats
import matplotlib.pyplot as plt

BASE_DIR = Path(os.environ.get('RWH_PUBLIC_BASE', r'c:/CHRISP/Regional_RWH')).resolve()
AOI_PATH = Path(os.environ.get('RWH_AOI_PATH', str(BASE_DIR / 'data' / 'aoi.shp')))
CHIRPS_DIR = Path(os.environ.get('RWH_CHIRPS_DIR', str(BASE_DIR / 'data' / 'chirps_daily')))
OUTPUT_DIR = Path(os.environ.get('RWH_OUTPUT_DIR', str(BASE_DIR / 'output_public')))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = OUTPUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RAIN_VAR_CANDIDATES = ['precip', 'precipitation', 'rain']
DIST_NAMES = ['gamma', 'lognorm', 'weibull_min', 'norm']
QUANTILE_LEVELS = [0.10, 0.50, 0.90]
MIN_WET_DAYS = 365

print('Configuration ready.')
print(f'AOI_PATH = {AOI_PATH}')
print(f'CHIRPS_DIR = {CHIRPS_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# 2. Load area of interest and identify CHIRPS pixels
if not AOI_PATH.exists():
    raise FileNotFoundError(f'Boundary file not found: {AOI_PATH}')

chirps_files = sorted(CHIRPS_DIR.glob('*.nc'))
if not chirps_files:
    raise FileNotFoundError(f'No NetCDF files found in: {CHIRPS_DIR}')

aoi_gdf = gpd.read_file(AOI_PATH)
if aoi_gdf.crs is None:
    aoi_gdf = aoi_gdf.set_crs('EPSG:4326')
else:
    aoi_gdf = aoi_gdf.to_crs('EPSG:4326')
aoi_boundary = aoi_gdf.dissolve()

with xr.open_dataset(chirps_files[0]) as sample_ds:
    if 'lat' in sample_ds.coords and 'lon' in sample_ds.coords:
        lat_name, lon_name = 'lat', 'lon'
    elif 'latitude' in sample_ds.coords and 'longitude' in sample_ds.coords:
        lat_name, lon_name = 'latitude', 'longitude'
    elif 'y' in sample_ds.coords and 'x' in sample_ds.coords:
        lat_name, lon_name = 'y', 'x'
    else:
        raise KeyError(f'Could not detect coordinate names: {list(sample_ds.coords.keys())}')

    rain_var = next((name for name in RAIN_VAR_CANDIDATES if name in sample_ds.data_vars), None)
    if rain_var is None:
        raise KeyError(f'Could not find a rainfall variable in {chirps_files[0].name}')

    lat_vals = sample_ds[lat_name].values
    lon_vals = sample_ds[lon_name].values

lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)
grid_points = gpd.GeoDataFrame({'geometry': [Point(lon, lat) for lon, lat in zip(lon_grid.ravel(), lat_grid.ravel())]}, crs='EPSG:4326')
grid_points['lat'] = grid_points.geometry.y
grid_points['lon'] = grid_points.geometry.x
pixels_in_aoi = gpd.sjoin(grid_points, aoi_boundary, how='inner', predicate='within').drop_duplicates(subset=['lat', 'lon']).reset_index(drop=True)

print(f'Found {len(pixels_in_aoi)} CHIRPS pixels inside the selected area.')

In [ ]:
# 3. Build a cached daily rainfall cube for the selected pixels
cache_path = CACHE_DIR / 'daily_rainfall_cube.pkl'

if cache_path.exists():
    with open(cache_path, 'rb') as f:
        daily_all = pickle.load(f)
    print(f'Loaded rainfall cube from cache: {cache_path}')
else:
    daily_list = []
    lat_min, lat_max = pixels_in_aoi['lat'].min(), pixels_in_aoi['lat'].max()
    lon_min, lon_max = pixels_in_aoi['lon'].min(), pixels_in_aoi['lon'].max()

    for fp in chirps_files:
        with xr.open_dataset(fp) as ds:
            if 'lat' in ds.coords and 'lon' in ds.coords:
                lat_n, lon_n = 'lat', 'lon'
            elif 'latitude' in ds.coords and 'longitude' in ds.coords:
                lat_n, lon_n = 'latitude', 'longitude'
            elif 'y' in ds.coords and 'x' in ds.coords:
                lat_n, lon_n = 'y', 'x'
            else:
                continue

            rain_var = next((name for name in RAIN_VAR_CANDIDATES if name in ds.data_vars), None)
            if rain_var is None:
                continue

            ds_r = ds.rename({lat_n: 'lat', lon_n: 'lon'})
            lat_vals = ds_r['lat'].values
            lon_vals = ds_r['lon'].values
            lat_slice = slice(lat_min, lat_max) if lat_vals[0] <= lat_vals[-1] else slice(lat_max, lat_min)
            lon_slice = slice(lon_min, lon_max) if lon_vals[0] <= lon_vals[-1] else slice(lon_max, lon_min)
            crop = ds_r[rain_var].sel(lat=lat_slice, lon=lon_slice).compute()
            daily_list.append(crop)

    if not daily_list:
        raise RuntimeError('No rainfall data could be loaded.')

    daily_all = xr.concat(daily_list, dim='time').sortby('time')
    with open(cache_path, 'wb') as f:
        pickle.dump(daily_all, f)
    print(f'Cached rainfall cube to {cache_path}')

print(daily_all)

In [ ]:
# 4. Fit simple distributions and export rainfall quantiles
results = []

for _, row in pixels_in_aoi[['lat', 'lon']].drop_duplicates().iterrows():
    lat = float(row['lat'])
    lon = float(row['lon'])

    try:
        series = daily_all.sel(lat=lat, lon=lon, method='nearest').to_dataframe(name='rain_mm').reset_index()
    except Exception:
        continue

    data = series['rain_mm'].dropna().astype(float).values
    data = data[data >= 1.0]
    if data.size < MIN_WET_DAYS:
        continue

    fit_rows = []
    for dist_name in DIST_NAMES:
        dist = getattr(stats, dist_name)
        try:
            params = dist.fit(data, floc=0) if dist_name != 'norm' else dist.fit(data)
            ll = np.sum(dist.logpdf(data, *params))
            aic = 2 * len(params) - 2 * ll
            q_values = dist.ppf(QUANTILE_LEVELS, *params)
            fit_rows.append({
                'distribution': dist_name,
                'params': list(map(float, params)),
                'aic': float(aic),
                'p10': float(q_values[0]),
                'p50': float(q_values[1]),
                'p90': float(q_values[2])
            })
        except Exception:
            continue

    if not fit_rows:
        continue

    fit_df = pd.DataFrame(fit_rows).sort_values('aic').reset_index(drop=True)
    best = fit_df.iloc[0].to_dict()
    best['lat'] = lat
    best['lon'] = lon
    best['n_wet_days'] = int(data.size)
    best['mean_rain_mm'] = float(np.mean(data))
    best['median_rain_mm'] = float(np.median(data))
    results.append(best)

results_df = pd.DataFrame(results)
results_csv = TABLE_DIR / 'rainfall_distribution_summary.csv'
results_df.to_csv(results_csv, index=False)
print(f'Saved summary table to {results_csv}')

sample_plot = None
if not results_df.empty:
    sample = results_df.iloc[0]
    sample_series = daily_all.sel(lat=float(sample['lat']), lon=float(sample['lon']), method='nearest').to_dataframe(name='rain_mm').reset_index()['rain_mm'].dropna().astype(float).values
    sample_series = sample_series[sample_series >= 1.0]
    x = np.linspace(sample_series.min(), sample_series.max(), 200)
    plt.figure(figsize=(10, 6))
    plt.hist(sample_series, bins=30, density=True, alpha=0.4, color='gray', label='Observed')
    for dist_name in DIST_NAMES:
        dist = getattr(stats, dist_name)
        row_match = results_df[results_df['distribution'] == dist_name]
        if row_match.empty:
            continue
        params = row_match.iloc[0]['params']
        try:
            y = dist.pdf(x, *params)
            plt.plot(x, y, label=dist_name)
        except Exception:
            continue
    plt.legend()
    plt.xlabel('Daily rainfall (mm)')
    plt.ylabel('Density')
    plt.title('Sample rainfall distribution fit')
    sample_plot = PLOTS_DIR / 'sample_fit.png'
    plt.tight_layout()
    plt.savefig(sample_plot, dpi=150)
    plt.close()
    print(f'Saved sample plot to {sample_plot}')

results_df.head()